In [0]:
from common.SilverPipelineClass import SilverPipeline
from pyspark.sql import DataFrame
import pyspark.sql.functions as F

In [0]:
class FinanceiroSilverPipeline(SilverPipeline):
    def __init__(self):
        super().__init__(dominio='fin')

    def transform(
        self,
        df: DataFrame,
        date_cols: list = None,
        int_cols: list = None,
        decimal_cols: list = None,
        required_cols: list = None,       # Nulos SOMENTE nessas colunas dropam a linha
        nullable_cols: list = None,        # Colunas onde nulo é estruturalmente válido
        status_fix: dict = None,           # Typos em colunas de status: {'pagoS': 'pago'}
        normalize_doc_cols: list = None,   # CPF/CNPJ: remove máscara e reaplica padrão
    ) -> 'pyspark.sql.DataFrame':

        # Transformação da classe pai
        df = super().transform(df)

        # ================================================================
        # NULOS — Drop seletivo por colunas obrigatórias
        # Substitui o dropna() genérico do colega, que derruba linhas com
        # nulos estruturalmente válidos (ex: data_pagamento de pendentes,
        # observacoes opcionais, id_referencia do saldo inicial).
        # ================================================================
        if required_cols:
            df = df.dropna(subset=required_cols)

        # ================================================================
        # TRATAMENTO DE DATAS
        # ================================================================
        formats = ["yyyy-MM-dd", "dd-MM-yyyy", "dd/MM/yyyy"]

        if date_cols:
            for col in date_cols:
                df = df.withColumn(
                    f'{col}_limpa',
                    F.coalesce(*[F.try_to_date(F.col(col), fmt) for fmt in formats])
                )
                df = df.filter(F.col(f'{col}_limpa').isNotNull())
                df = df.drop(col).withColumnRenamed(f'{col}_limpa', col)

        # ================================================================
        # TRATAMENTO DE INTEIROS
        # ================================================================
        if int_cols:
            for col in int_cols:
                df = df.withColumn(
                    f'{col}_limpa',
                    F.regexp_replace(F.col(col).cast("string"), r"[^0-9]", "").cast("int")
                )
                df = df.filter(F.col(f'{col}_limpa').isNotNull())
                df = df.drop(col).withColumnRenamed(f'{col}_limpa', col)

        # ================================================================
        # TRATAMENTO DE DECIMAIS
        # ================================================================
        if decimal_cols:
            for col in decimal_cols:
                df = df.withColumn(
                    f'{col}_limpa',
                    F.regexp_replace(F.col(col).cast("string"), r"[^0-9,.]", "")
                )
                df = df.withColumn(
                    f'{col}_limpa',
                    F.regexp_replace(F.col(f'{col}_limpa'), ",", ".").cast("decimal(18,2)")
                )
                df = df.filter(F.col(f'{col}_limpa').isNotNull())
                df = df.drop(col).withColumnRenamed(f'{col}_limpa', col)

        # ================================================================
        # CORREÇÃO DE TYPOS EM STATUS
        # Cobre casos como 'pagoS' -> 'pago' encontrado em pagamento_funcionarios.
        # Aplica lower() + strip() e depois o mapeamento de correções.
        # ================================================================
        if status_fix:
            for col, fix_map in status_fix.items():
                # Normaliza espaços e caixa antes de corrigir
                df = df.withColumn(col, F.trim(F.lower(F.col(col))))
                for wrong, correct in fix_map.items():
                    df = df.withColumn(
                        col,
                        F.when(F.col(col) == wrong.lower(), correct)
                         .otherwise(F.col(col))
                    )

        # ================================================================
        # NORMALIZAÇÃO DE CPF E CNPJ
        # Remove qualquer máscara existente (pontos, barras, hífens)
        # e reaplica o padrão correto com base no comprimento do número.
        #   CPF  → 11 dígitos → ###.###.###-##
        #   CNPJ → 14 dígitos → ##.###.###/####-##
        # Linhas onde o número limpo não tem 11 ou 14 dígitos são
        # marcadas como null (dado inválido/irrecuperável).
        # ================================================================
        if normalize_doc_cols:
            for col in normalize_doc_cols:
                # 1. Extrai só os dígitos
                digits = F.regexp_replace(F.col(col).cast("string"), r"[^0-9]", "")

                # 2. Aplica máscara conforme comprimento
                cpf_mask = F.concat(
                    F.substring(digits, 1, 3), F.lit("."),
                    F.substring(digits, 4, 3), F.lit("."),
                    F.substring(digits, 7, 3), F.lit("-"),
                    F.substring(digits, 10, 2)
                )
                cnpj_mask = F.concat(
                    F.substring(digits, 1, 2),  F.lit("."),
                    F.substring(digits, 3, 3),  F.lit("."),
                    F.substring(digits, 6, 3),  F.lit("/"),
                    F.substring(digits, 9, 4),  F.lit("-"),
                    F.substring(digits, 13, 2)
                )

                df = df.withColumn(
                    col,
                    F.when(F.length(digits) == 11, cpf_mask)
                     .when(F.length(digits) == 14, cnpj_mask)
                     .otherwise(F.lit(None))  # Inválido: nem CPF nem CNPJ
                )

        return df